In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from scipy.stats import pearsonr

import warnings
warnings.filterwarnings("ignore")

import os
os.getcwd()

#loading dtata set
news_df = pd.read_csv("../data/raw/raw_analyst_ratings.csv")
stock_df = pd.read_csv("../data/raw/AAPL.csv")
news_df.head()
stock_df.head()

#DATA ALIGNMENT
# DATA ALIGNMENT

# Convert news dates safely
news_df['date'] = pd.to_datetime(
    news_df['date'],
    format='mixed',
    utc=True,
    errors='coerce'
)

# Convert stock dates
stock_df['Date'] = pd.to_datetime(
    stock_df['Date'],
    errors='coerce'
)

# Drop invalid rows
news_df = news_df.dropna(subset=['date'])
stock_df = stock_df.dropna(subset=['Date'])

# Remove timezone info
news_df['date'] = news_df['date'].dt.tz_localize(None)

# Normalize timestamps
news_df['trading_date'] = news_df['date'].dt.normalize()

# Preview
print(news_df[['date', 'trading_date']].head())
#HANDLE WEEKEND
trading_days = stock_df['Date'].sort_values().tolist()

def align_to_trading_day(news_date):
    for day in trading_days:
        if day >= news_date:
            return day
    return np.nan

news_df['trading_date'] = news_df['date'].apply(align_to_trading_day)
#SENTINAL ANALYSIS
analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    return analyzer.polarity_scores(str(text))['compound']

news_df['sentiment_score'] = news_df['headline'].apply(get_sentiment)
news_df[['headline', 'sentiment_score']].head()
#AGREGATE DAILY SENTINAL
daily_sentiment = (
    news_df.groupby('trading_date')['sentiment_score']
    .mean()
    .reset_index()
)

daily_sentiment.rename(columns={
    'trading_date': 'Date',
    'sentiment_score': 'avg_sentiment'
}, inplace=True)

#CALCULATE DAILY STOCK
stock_df = stock_df.sort_values('Date')

stock_df['daily_return'] = (
    stock_df['Close'].pct_change() * 100
)
stock_df[['Date', 'Close', 'daily_return']].head()

#MERGAR SENTINAL TO DTAT
merged_df = pd.merge(
    stock_df,
    daily_sentiment,
    on='Date',
    how='inner'
)

merged_df.dropna(inplace=True)
#CORELLATIO ANALYSIS
corr, p_value = pearsonr(
    merged_df['avg_sentiment'],
    merged_df['daily_return']
)

print("Pearson Correlation:", corr)
print("P-value:", p_value)
#scaterplot
plt.figure(figsize=(8,6))

sns.scatterplot(
    data=merged_df,
    x='avg_sentiment',
    y='daily_return'
)

plt.title(f"Sentiment vs Stock Return\nCorrelation = {corr:.3f}")
plt.xlabel("Average Daily Sentiment")
plt.ylabel("Daily Stock Return (%)")

plt.axhline(0, color='gray', linestyle='--')
plt.axvline(0, color='gray', linestyle='--')

plt.show()
#sentiment chategoeey classifictaion

def classify_sentiment(score):
    if score > 0.05:
        return 'Positive'
    elif score < -0.05:
        return 'Negative'
    else:
        return 'Neutral'

merged_df['sentiment_category'] = (
    merged_df['avg_sentiment']
    .apply(classify_sentiment)
)
#avaerage return by chatgory
category_returns = (
    merged_df.groupby('sentiment_category')['daily_return']
    .mean()
    .reset_index()
)
#barchart visualization
plt.figure(figsize=(7,5))

sns.barplot(
    data=category_returns,
    x='sentiment_category',
    y='daily_return',
    palette='viridis'
)

plt.title("Average Daily Return by Sentiment Category")
plt.xlabel("Sentiment Category")
plt.ylabel("Average Return (%)")

plt.show()


                 date trading_date
0 2020-06-05 14:30:54   2020-06-05
1 2020-06-03 14:45:20   2020-06-03
2 2020-05-26 08:30:07   2020-05-26
3 2020-05-22 16:45:06   2020-05-22
4 2020-05-22 15:38:59   2020-05-22


In [ ]:
print(news_df.columns)